# Phase 9: Model lab — hyperparameters x ensembles x post-processing

Verdicts carried in: the v2 mechanisms are **dropped** (dZ coupling dead even
gated — corr(R², delta) = −0.04; smoothed-GR had a scale-mismatch bug, retained
here only as two fixed both-sides-smoothed guard configs). The lab takes the
three mechanisms that earned NNLS weight in notebook 7 — **particle filter,
beam search, hold** — and works the three lever classes systematically:

- **A. Hyperparameters** — combination sweeps around the v1 winners (nb6 was
  one-at-a-time; interactions like spread x momentum were never tested), plus a
  proper beam grid and seed-ensembles of the top two PF configs.
- **B. Ensemble permutations** — brute-force subset enumeration over the cached
  predictions (every 2^K−1 uniform blend), robust combiners (median),
  inverse-RMSE weights, NNLS — subset choice made **per fold**.
- **C. Post-processing** — hold-weight, seam anchoring, rolling-median
  smoothing, slew-rate clipping — every parameter fitted **out-of-fold**.

Default runs on a seeded 250-well sample (~45 min); set `WELL_SAMPLE = None`
for the full 773 before final conclusions. An aligned v1 reference is computed
on the *same* wells so comparisons are apples-to-apples.

## Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import json, sys, time
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("../src").resolve()))
from rogii_wellbore import clean  # noqa: E402

cfg = clean.load_config("../data/interim/clean_config.json")
CLEAN_DIR = Path("../data/interim/clean")
PRED_DIR_V3 = Path("../data/interim/pf_preds_v3"); PRED_DIR_V3.mkdir(parents=True, exist_ok=True)
PRED_DIR_V1 = Path("../data/interim/pf_preds")       # nb6 cache, for aligned reference
REAL_EVAL_FRAC = 0.73
N_FOLDS = 5
WELL_SAMPLE = 250          # None = all 773

def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))

def tail_mask(n, frac):
    k = int(round(n * frac)); m = np.zeros(n, bool)
    if k: m[n - k:] = True
    return m

def load_pair(wid):
    hz = pd.read_csv(CLEAN_DIR / "train" / f"{wid}__horizontal_well.csv",
                     dtype={"well_id": str}).sort_values("MD").reset_index(drop=True)
    tw = pd.read_csv(CLEAN_DIR / "train" / f"{wid}__typewell.csv",
                     dtype={"well_id": str}).reset_index(drop=True)
    return hz, tw

ALL_WELLS = sorted(p.name.split("__")[0]
                   for p in (CLEAN_DIR / "train").glob("*__horizontal_well.csv"))
if WELL_SAMPLE and WELL_SAMPLE < len(ALL_WELLS):
    WELLS = sorted(np.random.default_rng(123).choice(ALL_WELLS, WELL_SAMPLE, replace=False))
else:
    WELLS = ALL_WELLS
print(f"{len(WELLS)} wells in lab sample (of {len(ALL_WELLS)})")

## Engines (v1-faithful PF + beam; fixed both-sides GR smoothing as guard configs)

In [ ]:
def _prep_well(hz, tw, frac):
    tw_s = tw.sort_values("TVT")
    P = dict(
        twt=tw_s["TVT"].values.astype(float),
        twg=tw_s["GR"].ffill().bfill().values.astype(float),
        tvt=hz["TVT"].values.astype(float),
        Z=hz["Z"].values.astype(float), MD=hz["MD"].values.astype(float),
        gr=pd.Series(hz["GR"].values).interpolate(limit_direction="both")
            .fillna(90.0).values)
    m = tail_mask(len(hz), frac)
    P["kn"] = np.where(~m)[0]; P["ev"] = np.where(m)[0]
    return P

def run_pf(P, N=500, spread=4.5, MOM=0.998, VN=0.002, PN=0.005,
           rate_win=30, gs_max=60.0, seed=42, RESAMP=0.5, RP=0.1, RR=0.001,
           gr_wt=0.0, lat_sm=15, tw_sm=3):
    twt, twg, tvt, Z, MD, gr = (P[k] for k in ("twt", "twg", "tvt", "Z", "MD", "gr"))
    kn, ev = P["kn"], P["ev"]; last = kn[-1]
    gs = float(np.clip(np.nanstd(gr[kn] - np.interp(tvt[kn], twt, twg)), 10.0, gs_max))
    if gr_wt > 0:
        gr_s = pd.Series(gr).rolling(lat_sm, min_periods=1).mean().values      # causal
        twg_s = pd.Series(twg).rolling(tw_sm, min_periods=1, center=True).mean().values
    tl = kn[-rate_win:]
    dt = np.diff(tvt[tl]); dz = np.diff(Z[tl]); dm = np.diff(MD[tl]); ok = dm > 0
    ir = float(np.median((dt + dz)[ok] / dm[ok])) if ok.sum() >= 3 else 0.0
    rng = np.random.default_rng(seed)
    pos = (tvt[last] + Z[last]) + spread * rng.standard_normal(N)
    rate = ir + 0.01 * rng.standard_normal(N)
    w = np.ones(N) / N
    out = np.empty(len(ev)); prev = MD[last]
    lo, hi = twt[0] - 100, twt[-1] + 100
    for i, idx in enumerate(ev):
        dmS = max(MD[idx] - prev, 1.0)
        rate = MOM * rate + VN * rng.standard_normal(N)
        pos = pos + rate * dmS + PN * rng.standard_normal(N)
        tvt_p = np.clip(pos - Z[idx], lo, hi); pos = tvt_p + Z[idx]
        g = gr[idx]
        if np.isfinite(g):
            d2 = ((g - np.interp(tvt_p, twt, twg)) / gs) ** 2
            if gr_wt > 0:
                d2 = (1 - gr_wt) * d2 + gr_wt * \
                     ((gr_s[idx] - np.interp(tvt_p, twt, twg_s)) / gs) ** 2
            w = w * np.maximum(np.exp(-0.5 * np.minimum(d2, 600.0)), 1e-300)
            s = w.sum(); w = w / s if s > 0 else np.ones(N) / N
        if 1.0 / np.sum(w * w) < RESAMP * N:
            ci = np.clip(np.searchsorted(np.cumsum(w),
                 (np.arange(N) + rng.uniform(0, 1)) / N), 0, N - 1)
            pos = pos[ci] + RP * rng.standard_normal(N)
            rate = rate[ci] + RR * rng.standard_normal(N)
            w = np.ones(N) / N
        out[i] = np.sum(w * (pos - Z[idx])); prev = MD[idx]
    return out

def run_beam(P, BS=10, mc=20.0, es=144.0, smooth=2):
    twt, twg, tvt, MD, gr = (P[k] for k in ("twt", "twg", "tvt", "MD", "gr"))
    kn, ev = P["kn"], P["ev"]
    g = pd.Series(gr).rolling(smooth, min_periods=1, center=True).mean().values \
        if smooth > 1 else gr
    si = int(np.argmin(np.abs(twt - tvt[kn[-1]]))); beams = {si: 0.0}; hist = []
    for i in ev:
        gv = g[i]; cand = {}
        for idx, cost in beams.items():
            for d in (-2, -1, 0, 1, 2):
                ni = idx + d
                if ni < 0 or ni >= len(twt): continue
                tot = cost + (gv - twg[ni]) ** 2 / es + mc * abs(d)
                if ni not in cand or tot < cand[ni][0]: cand[ni] = (tot, idx)
        top = sorted(cand.items(), key=lambda kv: kv[1][0])[:BS]
        hist.append({ni: p for ni, (c, p) in top}); beams = {ni: c for ni, (c, p) in top}
    best = min(beams, key=beams.get); path = [best]
    for hm in reversed(hist[1:]): best = hm.get(best, best); path.append(best)
    return twt[np.array(path[::-1])]

## Part A — hyperparameter sweeps (combination grid, resumable cache)

In [ ]:
PF_CONFIGS = {
    "a_base":        dict(),
    "a_sp2":         dict(spread=2.0),
    "a_sp3":         dict(spread=3.0),
    "a_mom997":      dict(MOM=0.997),
    "a_mom999":      dict(MOM=0.999),
    "a_vn004":       dict(VN=0.004),
    "a_rp005":       dict(RP=0.05),
    "a_rp02":        dict(RP=0.2),
    "a_resamp07":    dict(RESAMP=0.7),
    "a_sp2_mom999":  dict(spread=2.0, MOM=0.999),
    "a_sp2_vn004":   dict(spread=2.0, VN=0.004),
    "a_grboth_15_3": dict(gr_wt=0.3, lat_sm=15, tw_sm=3),
    "a_grboth_25_5": dict(gr_wt=0.3, lat_sm=25, tw_sm=5),
}
SEED_ENS = {"ens_base": "a_base", "ens_sp2": "a_sp2"}
SEEDS = (42, 7, 2024, 99, 1234)
BEAM_CONFIGS = {
    "bm_cons":   dict(BS=10, mc=20.0, es=144.0, smooth=2),
    "bm_loose":  dict(BS=10, mc=8.0,  es=64.0,  smooth=2),
    "bm_vcons":  dict(BS=8,  mc=35.0, es=220.0, smooth=1),
    "bm_mid":    dict(BS=10, mc=12.0, es=100.0, smooth=2),
    "bm_wide":   dict(BS=12, mc=20.0, es=144.0, smooth=3),
    "bm_tight":  dict(BS=10, mc=28.0, es=180.0, smooth=2),
    "bm_sharp":  dict(BS=8,  mc=16.0, es=144.0, smooth=1),
    "bm_soft":   dict(BS=10, mc=20.0, es=80.0,  smooth=2),
}
COMP_KEYS = list(PF_CONFIGS) + list(SEED_ENS) + list(BEAM_CONFIGS)

t0 = time.time(); done = cached = 0
rows = []
for wid in WELLS:
    assert isinstance(wid, str)
    out_f = PRED_DIR_V3 / f"{wid}.npz"
    try:
        hz, tw = load_pair(wid)
    except Exception:
        continue
    if "TVT" not in hz or hz["TVT"].isna().all(): continue
    nrow = len(hz)
    if int(round(nrow * REAL_EVAL_FRAC)) < 20 or nrow - int(round(nrow * REAL_EVAL_FRAC)) < 20:
        continue
    P = _prep_well(hz, tw, REAL_EVAL_FRAC)
    true = P["tvt"][P["ev"]]; hold = np.full(len(P["ev"]), P["tvt"][P["kn"][-1]])
    preds = None
    if out_f.exists():
        z = np.load(out_f)
        if all(k in z.files for k in COMP_KEYS):
            preds = {k: z[k].astype(float) for k in COMP_KEYS}
            cached += 1
        else:
            out_f.unlink()
    if preds is None:
        preds = {name: run_pf(P, **kw) for name, kw in PF_CONFIGS.items()}
        for ens_name, base_name in SEED_ENS.items():
            kw = PF_CONFIGS[base_name]
            runs = [preds[base_name]] + [run_pf(P, **kw, seed=s) for s in SEEDS[1:]]
            preds[ens_name] = np.mean(runs, axis=0)
        for name, kw in BEAM_CONFIGS.items():
            preds[name] = run_beam(P, **kw)
        np.savez_compressed(out_f, true=true.astype(np.float32),
                            hold=hold.astype(np.float32),
                            **{k: v.astype(np.float32) for k, v in preds.items()})
        done += 1
    row = {"well": wid, "floor": rmse(hold, true)}
    for k in COMP_KEYS: row[k] = rmse(preds[k], true)
    rows.append(row)
    if done and done % 25 == 0:
        print(f"  {done} computed / {cached} cached  [{(time.time()-t0)/60:.1f} min]")
solo = pd.DataFrame(rows)
print(f"Part A ready: {len(solo)} wells ({done} computed, {cached} cached)  "
      f"[{(time.time()-t0)/60:.1f} min]")
print()
print("solo ranking (per-well mean):")
print(solo.drop(columns=["well"]).mean().sort_values().round(3).head(14))

## Part B — ensemble permutations (per-fold subset selection, robust combiners)

In [ ]:
from scipy.optimize import nnls

wells3, P3 = [], {}
for wid in solo["well"]:
    z = np.load(PRED_DIR_V3 / f"{wid}.npz")
    P3[wid] = {k: z[k].astype(float) for k in z.files}
    wells3.append(wid)
rng = np.random.default_rng(0)
fold_of = dict(zip(sorted(wells3), rng.permutation(len(wells3)) % N_FOLDS))

def pooled_rmse(pv):
    e = np.concatenate([pv[w] - P3[w]["true"] for w in pv])
    return float(np.sqrt(np.mean(e ** 2)))
def per_well_mean(pv):
    return float(np.mean([rmse(pv[w], P3[w]["true"]) for w in pv]))

TOPK = 10
solo_rank = solo.drop(columns=["well", "floor"]).mean().sort_values()
CAND = list(solo_rank.index[:TOPK]) + ["hold"]
print("candidate pool:", CAND)

def stacked(ws):
    X = np.vstack([np.column_stack([P3[w][k] for k in CAND]) for w in ws])
    y = np.concatenate([P3[w]["true"] for w in ws])
    return X, y

oof_subset, picks = {}, []
for f in range(N_FOLDS):
    tr = [w for w in wells3 if fold_of[w] != f]; va = [w for w in wells3 if fold_of[w] == f]
    Xtr, ytr = stacked(tr)
    best_r, best_sel = np.inf, None
    for mask in range(1, 2 ** len(CAND)):
        sel = [i for i in range(len(CAND)) if mask >> i & 1]
        r = float(np.sqrt(np.mean((Xtr[:, sel].mean(1) - ytr) ** 2)))
        if r < best_r: best_r, best_sel = r, sel
    picks.append([CAND[i] for i in best_sel])
    for w in va:
        oof_subset[w] = np.column_stack([P3[w][CAND[i]] for i in best_sel]).mean(1)
print("per-fold subset picks:")
for f, p in enumerate(picks): print(f"  fold {f}: {p}")

uni_all = {w: np.mean([P3[w][k] for k in CAND if k != "hold"], axis=0) for w in wells3}
med_all = {w: np.median(np.column_stack([P3[w][k] for k in CAND]), axis=1) for w in wells3}

oof_nnls, fold_w = {}, []
for f in range(N_FOLDS):
    tr = [w for w in wells3 if fold_of[w] != f]; va = [w for w in wells3 if fold_of[w] == f]
    X, y = stacked(tr)
    if len(y) > 300_000:
        idx = np.random.default_rng(f).choice(len(y), 300_000, replace=False)
        X, y = X[idx], y[idx]
    wts, _ = nnls(X, y); s = wts.sum(); wts = wts / s if s > 0 else np.ones(len(CAND)) / len(CAND)
    fold_w.append(wts)
    for w in va:
        oof_nnls[w] = np.column_stack([P3[w][k] for k in CAND]) @ wts

floor_pv = {w: P3[w]["hold"] for w in wells3}
print()
print(f"{'stage':18s} {'pooled':>8s} {'per-well':>9s}")
for name, pv in [("floor", floor_pv), ("uniform_top10", uni_all),
                 ("median_top10", med_all), ("subset_oof", oof_subset),
                 ("nnls_oof", oof_nnls)]:
    print(f"{name:18s} {pooled_rmse(pv):8.3f} {per_well_mean(pv):9.3f}")
wbar = np.mean(fold_w, axis=0)
print()
print("mean NNLS weights:", {k: round(float(v), 3) for k, v in zip(CAND, wbar) if v > 0.02})

if PRED_DIR_V1.exists():
    V1K = ["pf_spread2", "pf_seedens", "pf_base", "pf_N300", "pf_spread8", "beam_cons"]
    v1 = {}
    for w in wells3:
        f1 = PRED_DIR_V1 / f"{w}.npz"
        if f1.exists():
            z1 = np.load(f1)
            if all(k in z1.files for k in V1K):
                v1[w] = np.mean([z1[k].astype(float) for k in V1K], axis=0)
    if v1:
        e = np.concatenate([v1[w] - P3[w]["true"] for w in v1])
        pw = float(np.mean([rmse(v1[w], P3[w]["true"]) for w in v1]))
        print()
        print(f"aligned v1 uniform reference ({len(v1)} wells): "
              f"pooled {float(np.sqrt(np.mean(e**2))):.3f} | per-well {pw:.3f}")

## Part C — post-processing (every parameter out-of-fold)

Applied to the Part-B OOF subset blend. Families: hold-weight, seam anchor
(exponential pull toward the last known TVT near the boundary), rolling-median
smoothing, slew-rate clipping (per-row |dTVT| capped at a multiple of the
**known zone's** p99 — never eval truth). Per fold: pick the best parameter on
training wells, apply to validation. A do-nothing value is always a candidate,
so a family can win by doing nothing.

In [ ]:
BASE_PV = oof_subset

known_p99 = {}
for w in wells3:
    hz, _ = load_pair(w)
    tvt = hz["TVT"].values.astype(float)
    kn = np.where(~tail_mask(len(hz), REAL_EVAL_FRAC))[0]
    known_p99[w] = float(np.quantile(np.abs(np.diff(tvt[kn])), 0.99))

def pp_hold(pv, w_hold):
    return {w: (1 - w_hold) * pv[w] + w_hold * P3[w]["hold"] for w in pv}
def pp_anchor(pv, tau):
    out = {}
    for w in pv:
        p = pv[w]; t = np.arange(len(p), dtype=float)
        out[w] = p + (P3[w]["hold"][0] - p[0]) * np.exp(-t / tau)
    return out
def pp_median(pv, win):
    return {w: pd.Series(pv[w]).rolling(int(win), center=True, min_periods=1)
                 .median().values for w in pv}
def pp_slew(pv, mult):
    out = {}
    for w in pv:
        p = pv[w]; mx = known_p99[w] * mult
        d = np.clip(np.diff(p), -mx, mx)
        out[w] = np.concatenate([[p[0]], p[0] + np.cumsum(d)])
    return out

FAMILIES = {
    "hold_w":  ([0.0, 0.05, 0.1, 0.15, 0.2, 0.3], pp_hold, 0.0),
    "anchor":  ([0, 50, 200, 800, 3200], pp_anchor, 0),
    "med_win": ([1, 11, 41, 101], pp_median, 1),
    "slew":    ([0, 2.0, 4.0], pp_slew, 0),
}

def fold_pooled(pv, ws):
    e = np.concatenate([pv[w] - P3[w]["true"] for w in ws])
    return float(np.sqrt(np.mean(e ** 2)))

print(f"base (subset_oof): pooled {pooled_rmse(BASE_PV):.3f}")
results_pp = {}
for fam, (grid, fn, noop) in FAMILIES.items():
    oof = {}; chosen = []
    for f in range(N_FOLDS):
        tr = [w for w in wells3 if fold_of[w] != f]; va = [w for w in wells3 if fold_of[w] == f]
        best_p, best_r = noop, np.inf
        for prm in grid:
            cand = {w: BASE_PV[w] for w in tr} if prm == noop \
                   else fn({w: BASE_PV[w] for w in tr}, prm)
            r = fold_pooled(cand, tr)
            if r < best_r: best_r, best_p = r, prm
        chosen.append(best_p)
        out = {w: BASE_PV[w] for w in va} if best_p == noop \
              else fn({w: BASE_PV[w] for w in va}, best_p)
        for w in va: oof[w] = out[w]
    results_pp[fam] = (pooled_rmse(oof), chosen)
    print(f"  {fam:8s} OOF pooled {results_pp[fam][0]:8.3f}  per-fold picks {chosen}")

## Final table + recipe v3

In [ ]:
final = {
    "floor": pooled_rmse(floor_pv),
    "uniform_top10": pooled_rmse(uni_all),
    "median_top10": pooled_rmse(med_all),
    "subset_oof": pooled_rmse(oof_subset),
    "nnls_oof": pooled_rmse(oof_nnls),
}
for fam, (r, _) in results_pp.items():
    final[f"subset+{fam}"] = r
print(f"{'stage':22s} {'pooled':>8s}")
for k, v in sorted(final.items(), key=lambda kv: kv[1]):
    print(f"{k:22s} {v:8.3f}")

RECIPE_V3 = {
    "pf_configs": {k: v for k, v in PF_CONFIGS.items()},
    "beam_configs": {k: v for k, v in BEAM_CONFIGS.items()},
    "seed_ens": SEED_ENS, "seeds": list(SEEDS),
    "candidate_pool": CAND,
    "per_fold_subsets": picks,
    "nnls_mean_weights": {k: float(v) for k, v in zip(CAND, wbar)},
    "postproc_oof": {k: {"pooled": float(v[0]), "fold_picks": [str(p) for p in v[1]]}
                     for k, v in results_pp.items()},
    "well_sample": len(wells3),
}
with open("../data/interim/recipe_v3.json", "w") as f:
    json.dump(RECIPE_V3, f, indent=2, default=str)
best_name = min(final, key=final.get)
print()
print(f"WINNER: {best_name} ({final[best_name]:.3f})")
print("saved recipe_v3.json")

## Reading & next

- **Part A**: do any combo configs beat `ens_base`/`ens_sp2`? Do the fixed
  both-sides `grboth` guards redeem the mechanism, or is it dropped for good?
- **Part B**: subset_oof vs uniform vs median vs NNLS — and per-fold subset
  stability (same components across folds = trustworthy composition).
- **Part C**: families picking non-trivial parameters consistently across folds
  = real effect; do-nothing picks = the family adds nothing.
- Compare everything against the **aligned v1 reference** printed in Part B.
- If the winner lands meaningfully below v1 on the full set, fold it into
  `submission.ipynb`; if it plateaus at v1's level, the engine is exhausted on
  single-well information and the next build is the **cross-well surface prior**.